Dataset repository link

https://huggingface.co/datasets/cornell-movie-review-data/rotten_tomatoes

In [5]:
# Install required deep learning and natural language processing libraries in the environment.
!pip install -q transformers datasets evaluate torch scikit-learn nltk

import os
import torch
import numpy as np
from datasets import load_dataset

# Load the official movie review dataset directly from its canonical repository path.
dataset = load_dataset("cornell-movie-review-data/rotten_tomatoes")

# Print a confirmation message along with the first training instance to verify the dataset structure.
print("Dataset loaded successfully!")
print(dataset['train'][0])

# Slice small, randomized subsets of the data to keep model training times short and manageable.
train_data = dataset['train'].shuffle(seed=42).select(range(2000))
test_data = dataset['test'].shuffle(seed=42).select(range(500))

README.md:   0%|          | 0.00/7.46k [00:00<?, ?B/s]

train.parquet:   0%|          | 0.00/699k [00:00<?, ?B/s]

validation.parquet:   0%|          | 0.00/90.0k [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/92.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Dataset loaded successfully!
{'text': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .', 'label': 1}


## BERT MODEL

In [6]:
import evaluate
from transformers import BertTokenizer, BertForSequenceClassification, TrainingArguments, Trainer

# Initialize the pre-trained uncased BERT tokenizer and sequence classification model from Hugging Face.
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# Define a tokenization utility to truncate and pad sentences to a uniform length of 64 tokens.
def tokenize_bert(examples):
    return bert_tokenizer(examples['text'], padding="max_length", truncation=True, max_length=64)

# Apply the tokenization utility across the split train and test datasets in fast parallel batches.
bert_train = train_data.map(tokenize_bert, batched=True)
bert_test = test_data.map(tokenize_bert, batched=True)

# Combine standard classification evaluation metrics including Accuracy, F1, Precision, and Recall.
clf_metrics = evaluate.combine(["accuracy", "f1", "precision", "recall"])

# Compute standard classification metrics by picking the highest logit probability as the prediction.
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return clf_metrics.compute(predictions=predictions, references=labels)

# Configure the hyperparameter settings, batch sizes, and epoch numbers for the BERT training run.
bert_args = TrainingArguments(
    output_dir="./bert_results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
)

# Instantiate the Hugging Face Trainer by linking the model, datasets, parameters, and metrics together.
bert_trainer = Trainer(
    model=bert_model,
    args=bert_args,
    train_dataset=bert_train,
    eval_dataset=bert_test,
    compute_metrics=compute_metrics,
)

# Train the BERT model on the text sequences and print its final out-of-sample evaluation metrics.
print("Training BERT...")
bert_trainer.train()
print("BERT Evaluation Metrics:", bert_trainer.evaluate())

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

...

Training BERT...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.414732,0.824000,0.805310,0.846512,0.767932
2,No log,0.413959,0.844000,0.831897,0.850220,0.814346
3,No log,0.491964,0.844000,0.834043,0.841202,0.827004


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
No log,0.491964,3,0.844000,0.834043,0.841202,0.827004


BERT Evaluation Metrics: {'eval_loss': 0.4919641613960266, 'eval_accuracy': 0.844, 'eval_f1': 0.8340425531914893, 'eval_precision': 0.8412017167381974, 'eval_recall': 0.8270042194092827}


## GPT MODEL

In [7]:
import math
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForLanguageModeling

# Instantiate the lightweight DistilGPT2 tokenizer and causal language model.
gpt_tokenizer = AutoTokenizer.from_pretrained('distilgpt2')
# Set the padding token to equal the end-of-sequence token since GPT models lack a default pad token.
gpt_tokenizer.pad_token = gpt_tokenizer.eos_token
gpt_model = AutoModelForCausalLM.from_pretrained('distilgpt2')

# Append the end-of-sequence token to reviews before truncating them to a uniform sequence length.
def tokenize_gpt(examples):
    # Add EOS token to the end of each text so the model learns to stop
    texts = [text + gpt_tokenizer.eos_token for text in examples['text']]
    return gpt_tokenizer(texts, truncation=True, max_length=64)

# Tokenize datasets and remove target labels since causal models only learn to predict the next word.
gpt_train = train_data.map(tokenize_gpt, batched=True, remove_columns=['label'])
gpt_test = test_data.map(tokenize_gpt, batched=True, remove_columns=['label'])

# Initialize a causal data collator that dynamically shifts inputs to handle language generation targets.
data_collator = DataCollatorForLanguageModeling(tokenizer=gpt_tokenizer, mlm=False)

# Define the training hyperparameters and set the evaluation strategy to run at the end of each epoch.
gpt_args = TrainingArguments(
    output_dir="./gpt_results",
    eval_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    num_train_epochs=3,
)

# Construct the language modeling trainer container using the modified autoregressive datasets.
gpt_trainer = Trainer(
    model=gpt_model,
    args=gpt_args,
    train_dataset=gpt_train,
    eval_dataset=gpt_test,
    data_collator=data_collator,
)

# Fine-tune the causal GPT model across the review texts.
print("Training GPT...")
gpt_trainer.train()

# Exponentiate the validation cross-entropy loss to evaluate the model's final word Perplexity score.
eval_results = gpt_trainer.evaluate()
perplexity = math.exp(eval_results['eval_loss'])
print(f"GPT Perplexity: {perplexity:.2f}")

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

Epoch,Training Loss,Validation Loss
1,No log,4.257953
2,4.374851,4.230138
3,4.374851,4.231155


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch
4.374851,4.231155,3


GPT Perplexity: 68.80


## Text-GAN Model

In [8]:
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# Assign execution to an active CUDA GPU backend if it is available in the runtime environment.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define the tensor dimensions, vocabulary limits, and batch sizes for the adversarial training loop.
vocab_size = 1000
embed_dim = 64
hidden_dim = 128
seq_len = 20
batch_size = 32

# Construct a Generator network that leverages Gumbel-Softmax to output differentiable text token probabilities.
class TextGenerator(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, noise, temperature=1.0):
        # noise shape: (batch, seq_len, embed_dim)
        lstm_out, _ = self.lstm(noise)
        logits = self.fc(lstm_out)
        # Gumbel-Softmax allows backprop through discrete text tokens!
        soft_tokens = F.gumbel_softmax(logits, tau=temperature, hard=True)
        return soft_tokens # shape: (batch, seq_len, vocab_size)

# Construct a Discriminator network that evaluates token streams and projects a real-versus-fake probability score.
class TextDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Linear(vocab_size, embed_dim) # Treat soft-tokens as embeddings
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, soft_tokens):
        x = self.embedding(soft_tokens)
        _, (h_n, _) = self.lstm(x)
        out = torch.sigmoid(self.fc(h_n[-1]))
        return out

# Initialize network weights, set up independent Adam optimizers, and instantiate Binary Cross Entropy loss.
netG = TextGenerator().to(device)
netD = TextDiscriminator().to(device)
optimizerG = optim.Adam(netG.parameters(), lr=0.001)
optimizerD = optim.Adam(netD.parameters(), lr=0.001)
criterion = nn.BCELoss()

# Execute a lightweight generative adversarial training loop across a fixed number of epochs.
print("Training Text-GAN...")
epochs = 5

for epoch in range(epochs):
    # Construct synthetic ground-truth real data representations using randomized uniform integer matrices.
    real_data = F.one_hot(torch.randint(0, vocab_size, (batch_size, seq_len)), num_classes=vocab_size).float().to(device)

    # Reset gradient histories for the Discriminator network prior to backward pass execution.
    optimizerD.zero_grad()

    # Pass the real matrix through the Discriminator and calculate its loss against target labels of 1.
    real_labels = torch.ones(batch_size, 1).to(device)
    out_real = netD(real_data)
    lossD_real = criterion(out_real, real_labels)

    # Synthesize fake distributions from random noise, detaching them to prevent early Generator gradient updates.
    noise = torch.randn(batch_size, seq_len, embed_dim).to(device)
    fake_data = netG(noise)
    fake_labels = torch.zeros(batch_size, 1).to(device)
    out_fake = netD(fake_data.detach()) # Detach so we don't backprop into G
    lossD_fake = criterion(out_fake, fake_labels)

    # Combine losses, compute backpropagation gradients, and step the Discriminator's optimizer weights.
    lossD = lossD_real + lossD_fake
    lossD.backward()
    optimizerD.step()

    # Reset gradient histories for the Generator network to prepare for adversarial text synthesis adjustments.
    optimizerG.zero_grad()

    # Compute adversarial generator loss by evaluating how successfully the generated outputs trick the Discriminator.
    out_fake_for_G = netD(fake_data)
    lossG = criterion(out_fake_for_G, real_labels)

    # Calculate optimization gradients for the Generator weights and apply the adjustment step.
    lossG.backward()
    optimizerG.step()

    # Measure real vs fake classification accuracy to monitor performance metrics.
    d_acc = ((out_real > 0.5).sum() + (out_fake < 0.5).sum()) / (batch_size * 2)

    # Print log metrics detailing training losses and accuracy updates at every scheduled step.
    if epoch % 1 == 0:
        print(f"Epoch [{epoch+1}/{epochs}] | Loss D: {lossD.item():.4f} | Loss G: {lossG.item():.4f} | D Acc: {d_acc:.4f}")

Training Text-GAN...
Epoch [1/5] | Loss D: 1.3877 | Loss G: 0.7155 | D Acc: 0.5000
Epoch [2/5] | Loss D: 1.3867 | Loss G: 0.7031 | D Acc: 0.5000
Epoch [3/5] | Loss D: 1.3863 | Loss G: 0.6916 | D Acc: 0.5000
Epoch [4/5] | Loss D: 1.3862 | Loss G: 0.6829 | D Acc: 0.5000
Epoch [5/5] | Loss D: 1.3863 | Loss G: 0.6782 | D Acc: 0.5000
